# ChemiAI — финальный воспроизводимый пайплайн

Ноутбук с финальным решением задачи регрессии по молекулярным дескрипторам. Для каждого соединения предсказываются **IC50**, **CC50** и **SI**.

Главная цель этого ноутбука — собрать итоговый сабмит **из одного пайплайна**, без чтения готовых submission-файлов.

## Описание задачи

Разработка лекарств требует оценки активности и токсичности соединений. В датасете представлены числовые молекулярные дескрипторы, по которым нужно предсказать три величины:

| Таргет | Смысл |
|--------|-------|
| **IC50 (mM)** | концентрация, при которой подавляется 50% активности вируса |
| **CC50 (mM)** | концентрация токсичности для 50% клеток |
| **SI** | индекс селективности, связанный с отношением `CC50 / IC50` |

Метрика соревнования — средний RMSE по трем таргетам:

$$\text{score} = \frac{RMSE(IC50) + RMSE(CC50) + RMSE(SI)}{3}$$

## План работы

1. **Настройка окружения** — импорт библиотек и фиксирование констант.
2. **Загрузка данных** — чтение `train`, `test`, `sample_submission` и базовые проверки.
3. **Clustering baseline** — построение базовой модели на исходных дескрипторах + `KMeans`-кластерах.
4. **Transductive neighbor** — построение локальной модели `PCA + KNN`, где preprocessing обучается на `train + test` без таргетов.
5. **Финальный ансамбль** — объединение двух моделей по таргетам.
6. **Сабмит** — сохранение `submission_transductive_neighbor_target_blend.csv` и проверка формата.

## 1. Настройка окружения

Импортируем библиотеки, задаем пути и фиксируем основные имена колонок. Все модели используют один `RANDOM_STATE`, чтобы результат был воспроизводимым.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TARGET_COLS = ["IC50, mM", "CC50, mM", "SI"]
SUBMISSION_COLS = ["IC50", "CC50", "SI"]
ID_COL = "index"

DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
BEST_SUBMISSION_PATH = Path("submission_transductive_neighbor_target_blend.csv")

## 2. Загрузка и подготовка данных

Читаем локальные файлы `train.csv`, `test.csv` и `sample_submission.csv`. Из признаков удаляем константные колонки: они не несут информации для моделей и могут мешать геометрическим методам.

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

feature_cols = [c for c in train.columns if c not in [ID_COL, *TARGET_COLS]]
X_train = train[feature_cols].copy()
y_train = train[TARGET_COLS].copy()
X_test = test[feature_cols].copy()

const_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
X_train = X_train.drop(columns=const_cols)
X_test = X_test.drop(columns=const_cols)

print("Train:", train.shape)
print("Test:", test.shape)
print("Features after constant drop:", X_train.shape[1])
print("Dropped constant columns:", len(const_cols))

## 3. Вспомогательные функции

Определяем метрику соревнования и функцию для transductive preprocessing. Она обучает unsupervised-шаги (`imputer`, `scaler`, `PCA`) на `train-fold + test`, но не использует test-таргеты.

In [ ]:
def competition_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    rmses = [np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])) for i in range(y_true.shape[1])]
    return float(np.mean(rmses))


def target_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> list[float]:
    return [float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))) for i in range(y_true.shape[1])]


def make_transductive_pca_features(
    X_fit_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    X_test_full: pd.DataFrame,
    n_components: int = 20,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Fit unsupervised preprocessing on train-fold + test, then transform train/valid/test."""
    fit_frame = pd.concat([X_fit_train, X_test_full], axis=0)

    imputer = SimpleImputer(strategy="median")
    fit_imputed = imputer.fit_transform(fit_frame)

    scaler = StandardScaler()
    fit_scaled = scaler.fit_transform(fit_imputed)

    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    pca.fit(fit_scaled)

    X_fit_pca = pca.transform(scaler.transform(imputer.transform(X_fit_train)))
    X_valid_pca = pca.transform(scaler.transform(imputer.transform(X_valid)))
    X_test_pca = pca.transform(scaler.transform(imputer.transform(X_test_full)))

    return X_fit_pca, X_valid_pca, X_test_pca

## 4. Clustering baseline

Строим baseline внутри ноутбука, без чтения готового `submission_clustering.csv`.

Конфигурация:

```text
SimpleImputer(median)
StandardScaler для KMeans
KMeans(k=4) -> one-hot cluster features
HistGradientBoostingRegressor по каждому таргету в log1p-шкале
5-fold усреднение test-предсказаний
```

Смысл этого блока — поймать дискретные режимы молекул. Кластерный признак помогает модели отличать локальные группы соединений, даже если отдельные дескрипторы сами по себе слабые.

In [ ]:
def build_clustering_features(
    X_fit: pd.DataFrame,
    X_apply: pd.DataFrame,
    n_clusters: int = 4,
) -> tuple[np.ndarray, np.ndarray]:
    imputer = SimpleImputer(strategy="median")
    X_fit_imp = imputer.fit_transform(X_fit)
    X_apply_imp = imputer.transform(X_apply)

    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit_imp)
    X_apply_scaled = scaler.transform(X_apply_imp)

    kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init="auto")
    fit_clusters = kmeans.fit_predict(X_fit_scaled)
    apply_clusters = kmeans.predict(X_apply_scaled)

    X_fit_aug = np.hstack([X_fit_imp, np.eye(n_clusters)[fit_clusters]])
    X_apply_aug = np.hstack([X_apply_imp, np.eye(n_clusters)[apply_clusters]])

    return X_fit_aug, X_apply_aug


kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

clustering_oof = np.zeros((len(X_train), len(TARGET_COLS)), dtype=float)
clustering_test_pred = np.zeros((len(X_test), len(TARGET_COLS)), dtype=float)

X_train_cluster_full, X_test_cluster_full = build_clustering_features(X_train, X_test, n_clusters=4)

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train_cluster_full), start=1):
    X_fit = X_train_cluster_full[train_idx]
    X_valid = X_train_cluster_full[valid_idx]
    y_fit = y_train.iloc[train_idx].values

    for target_idx, target_col in enumerate(TARGET_COLS):
        model = HistGradientBoostingRegressor(
            max_depth=6,
            learning_rate=0.03,
            max_iter=700,
            random_state=RANDOM_STATE,
        )
        model.fit(X_fit, np.log1p(y_fit[:, target_idx]))

        valid_pred = np.expm1(np.clip(model.predict(X_valid), 0, 12))
        test_pred = np.expm1(np.clip(model.predict(X_test_cluster_full), 0, 12))

        clustering_oof[valid_idx, target_idx] = np.clip(valid_pred, 0, None)
        clustering_test_pred[:, target_idx] += np.clip(test_pred, 0, None) / kf.n_splits

print("Clustering baseline OOF:", competition_score(y_train.values, clustering_oof))
print("Per-target RMSE:", np.round(target_rmse(y_train.values, clustering_oof), 5).tolist())

## 5. Transductive neighbor-модель

Параметры финальной neighbor-модели:

```text
PCA(n_components=20) + KNN(k=5, weights="distance")
```

Модель обучается в `log1p`-шкале по всем трем таргетам. В финальном сабмите используются только `CC50` и `SI` из этой модели.

Почему используется transductive preprocessing: `PCA` обучается на `train-fold + test`, поэтому пространство соседей лучше соответствует объектам, для которых реально нужен сабмит.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

transductive_oof = np.zeros((len(X_train), len(TARGET_COLS)), dtype=float)
transductive_test_pred = np.zeros((len(X_test), len(TARGET_COLS)), dtype=float)

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train), start=1):
    X_fit = X_train.iloc[train_idx]
    X_valid = X_train.iloc[valid_idx]
    y_fit = y_train.iloc[train_idx].values

    X_fit_pca, X_valid_pca, X_test_pca = make_transductive_pca_features(
        X_fit_train=X_fit,
        X_valid=X_valid,
        X_test_full=X_test,
        n_components=20,
    )

    model = KNeighborsRegressor(n_neighbors=5, weights="distance")
    model.fit(X_fit_pca, np.log1p(y_fit))

    valid_pred = np.expm1(model.predict(X_valid_pca))
    test_pred = np.expm1(model.predict(X_test_pca))

    transductive_oof[valid_idx] = np.clip(valid_pred, 0, None)
    transductive_test_pred += np.clip(test_pred, 0, None) / kf.n_splits

print("Transductive neighbor OOF:", competition_score(y_train.values, transductive_oof))
print("Per-target RMSE:", np.round(target_rmse(y_train.values, transductive_oof), 5).tolist())

## 6. Сборка финального сабмита

Смешиваем два предсказания, оба получены выше в этом же ноутбуке:

- `clustering_test_pred` — baseline на кластерных признаках;
- `transductive_test_pred` — локальная neighbor-модель.

Финальная формула:

```text
IC50 = 100% clustering
CC50 = 40% clustering + 60% transductive neighbor
SI   = 70% clustering + 30% transductive neighbor
```

`IC50` не трогаем, потому что neighbor-модель ухудшала этот таргет локально. Основной полезный вклад neighbor-модели идет через `CC50`.

In [ ]:
submission_best = sample_submission.copy()
submission_best["IC50"] = clustering_test_pred[:, TARGET_COLS.index("IC50, mM")]
submission_best["CC50"] = (
    0.40 * clustering_test_pred[:, TARGET_COLS.index("CC50, mM")]
    + 0.60 * transductive_test_pred[:, TARGET_COLS.index("CC50, mM")]
)
submission_best["SI"] = (
    0.70 * clustering_test_pred[:, TARGET_COLS.index("SI")]
    + 0.30 * transductive_test_pred[:, TARGET_COLS.index("SI")]
)

submission_best[SUBMISSION_COLS] = submission_best[SUBMISSION_COLS].clip(lower=0)
submission_best.to_csv(BEST_SUBMISSION_PATH, index=False)

print("Saved:", BEST_SUBMISSION_PATH)
print(submission_best.head())
print(submission_best.describe())

## 7. Проверка формата

Перед отправкой проверяем, что файл соответствует `sample_submission.csv`: правильные колонки, правильное число строк, нет пропусков и отрицательных предсказаний.

In [ ]:
# Финальные проверки формата сабмита.
submission_check = pd.read_csv(BEST_SUBMISSION_PATH)

assert list(submission_check.columns) == ["index", *SUBMISSION_COLS]
assert submission_check.shape == sample_submission.shape
assert submission_check[SUBMISSION_COLS].notna().all().all()
assert (submission_check[SUBMISSION_COLS] >= 0).all().all()
assert submission_check["index"].equals(sample_submission["index"])

print("Submission format is valid")
print("Rows:", len(submission_check))
print("Columns:", submission_check.columns.tolist())